# MobileNet / ViT — standard/robust comparison

Дообучает отдельного кандидата от выбранного standard-чемпиона. Для смены архитектуры достаточно изменить `MODEL`. Standard registry не изменяется; полный robust-кандидат может обновить только `registry/robust`.

In [ ]:
REPO_URL = "https://github.com/frest1ler/text-orientation-classification.git"
BRANCH = "main"
PROJECT_DIR = "/content/drive/MyDrive/text-orientation"
MODEL = "vit"  # vit | mobilenet
AUGMENTATION_PROFILE = "robust"  # standard | robust
QUICK_RUN = True
EPOCHS = 3
LEARNING_RATE = None  # None = architecture default
TRAIN_BATCH_SIZE = None  # ViT default: 16; MobileNet: 64
VALIDATION_BATCH_SIZE = None  # ViT default: 32; MobileNet: 128
NUM_WORKERS = 2
RESUME_TRAINING = True
RUN_TESTS = True
PROMOTE_ROBUST_CHAMPION = not QUICK_RUN and AUGMENTATION_PROFILE == "robust"

In [ ]:
import os, subprocess, sys
from pathlib import Path

repo_dir = Path("/content/text-orientation-classification")
if not repo_dir.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json, torch
MODEL_OPTIONS = {
    "mobilenet": {"name": "mobilenet_v3_large", "config": "configs/baseline.yaml", "lr": 2e-5, "train_batch": 64, "validation_batch": 128},
    "vit": {"name": "vit_b_16", "config": "configs/vit_b_16.yaml", "lr": 1e-5, "train_batch": 16, "validation_batch": 32},
}
if MODEL not in MODEL_OPTIONS:
    raise ValueError("MODEL must be 'vit' or 'mobilenet'")
model_option = MODEL_OPTIONS[MODEL]
model_name, config_path = model_option["name"], model_option["config"]
learning_rate = LEARNING_RATE or model_option["lr"]
train_batch_size = TRAIN_BATCH_SIZE or model_option["train_batch"]
validation_batch_size = VALIDATION_BATCH_SIZE or model_option["validation_batch"]
if AUGMENTATION_PROFILE not in {"standard", "robust"}:
    raise ValueError("AUGMENTATION_PROFILE must be 'standard' or 'robust'")
if not torch.cuda.is_available():
    raise RuntimeError("В Colab выберите Runtime → Change runtime type → GPU")
print("gpu:", torch.cuda.get_device_name(0))
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
from src.registry import select_champion
registry = Path(PROJECT_DIR) / "registry"
bundle = select_champion(registry, model_name)
initial_checkpoint = bundle.checkpoint_path
print({"model": model_name, "config": config_path, "initial_checkpoint": str(initial_checkpoint), "train_batch_size": train_batch_size})

In [ ]:
from IPython.display import Image as DisplayImage, display
preview = Path("/content/robust_preview.png")
subprocess.run([
    sys.executable, "-m", "scripts.preview_synthetic",
    "--augmentation-profile", AUGMENTATION_PROFILE,
    "--pairs", "8", "--epoch", "1", "--output", str(preview),
], check=True)
display(DisplayImage(filename=str(preview)))

In [ ]:
mode = "quick" if QUICK_RUN else "full"
run_name = f"{model_name}_{AUGMENTATION_PROFILE}_{mode}"
run_dir = Path("artifacts/experiments") / run_name
recovery_dir = Path(PROJECT_DIR) / "training/recovery" / AUGMENTATION_PROFILE / model_name / mode
command = [
    sys.executable, "-m", "scripts.train_robust",
    "--config", config_path,
    "--output-dir", str(run_dir),
    "--recovery-dir", str(recovery_dir),
    "--augmentation-profile", AUGMENTATION_PROFILE,
    "--initial-checkpoint", str(initial_checkpoint),
    "--epochs", str(2 if QUICK_RUN else EPOCHS),
    "--learning-rate", str(learning_rate),
    "--batch-size", str(train_batch_size),
    "--validation-batch-size", str(validation_batch_size),
    "--num-workers", str(NUM_WORKERS),
]
if QUICK_RUN:
    command += ["--train-base-samples", "2048", "--validation-base-samples", "512"]
if RESUME_TRAINING:
    command.append("--resume")
subprocess.run(command, check=True)

In [ ]:
import platform, shutil
from datetime import datetime, timezone

environment = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "git_commit": subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip(),
}
(run_dir / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
if not QUICK_RUN:
    subprocess.run([sys.executable, "-m", "scripts.calibrate", "--run-dir", str(run_dir), "--config", config_path], check=True)
    if PROMOTE_ROBUST_CHAMPION:
        subprocess.run([sys.executable, "-m", "scripts.promote_robust_champion", "--run-dir", str(run_dir), "--project-dir", PROJECT_DIR], check=True)
        subprocess.run([sys.executable, "-m", "scripts.compare_robust", "--project-dir", PROJECT_DIR, "--model", model_name, "--minimum-improvement", "0.005"], check=True)
runs_dir = Path(PROJECT_DIR) / "training/runs" / AUGMENTATION_PROFILE / model_name / mode
runs_dir.mkdir(parents=True, exist_ok=True)
archive = Path(shutil.make_archive(f"/content/{run_name}", "zip", root_dir=run_dir))
destination = runs_dir / archive.name
shutil.copy2(archive, destination)
print("Готово:", destination)
print("Standard registry не изменён.")